# 07 Real Portfolio — Stop-Loss / Re-entry Analysis

Applies the stop-loss and re-entry simulation from notebook 04 to **every position
in your real Deutsche Bank portfolio**, sourcing cost-basis data directly from the
broker PDF and fetching live market prices via Yahoo Finance.

**What this notebook does:**
1. Parses the DB PDF to extract real holdings (ISIN, shares, EUR cost basis per share)
2. Fetches current EUR prices for each position
3. For every position runs the full stop × bear-scenario cross-product
4. Shows a **portfolio ranking table** — which positions benefit most from a stop
5. Shows **per-position heatmaps** (same format as notebook 04) using real numbers

**Key question answered:** Given my actual gains and position sizes today,
which holdings are the best candidates for stop-loss protection, and at what trigger level?

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pdf_parser import parse_db_pdf
from portfolio_sim import (
    ECBProvider,
    YahooPriceProvider,
    fetch_current_prices,
    initialize_lots_from_holdings,
)
from tax_risk_sim import (
    build_bear_recovery_cases,
    build_bear_recovery_table,
    build_stop_benchmark,
    compare_stop_reentry_vs_hold,
    sell_today_baseline,
)

pd.options.display.float_format = "{:,.2f}".format

## Inputs

Edit `PDF_PATH` and `TICKER_MAP_PATH` to point at your local files. Use `data/private/ticker_map.json` for real mappings because it is gitignored; `data/examples/ticker_map_synthetic.json` is the committed demo map.
All other parameters use the same defaults as notebook 04.

In [ ]:
import json

# ── File paths ────────────────────────────────────────────────────────────────
PDF_PATH        = PROJECT_ROOT / "data/private/report.pdf"
TICKER_MAP_PATH = PROJECT_ROOT / "data/ticker_map.json"
REPORTING_DATE  = "2026-06-06"   # ISO date used for price lookup

# ── Tax & simulation parameters (match notebook 04 defaults) ─────────────────
CAPITAL_GAINS_TAX_RATE      = 0.26375   # German Abgeltungsteuer + Soli
STOP_LOSS_DROPS             = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30])
REENTRY_SLIPPAGE            = 0.05      # 5% above the bear low
TRANSACTION_COST_RATE       = 0.001     # 0.1% per leg
ALLOW_FRACTIONAL_SHARES     = False

# ── Bear scenario grid ────────────────────────────────────────────────────────
BEAR_START              = -0.05
BEAR_END                = -0.60
BEAR_STEP               = -0.05        # 5% steps → 12 columns (keeps heatmaps readable)
BEAR_RECOVERY_MULT      = 1.50
BEAR_MIN_RECOVERY       = 0.10
BEAR_MAX_RECOVERY       = 1.50
BEAR_BASE_PROB          = 0.70
BEAR_MIN_PROB           = 0.10

# ── Load ticker map ───────────────────────────────────────────────────────────
with open(TICKER_MAP_PATH) as f:
    _raw = json.load(f)
TICKER_MAP = {k: v for k, v in _raw.items() if not k.startswith("_")}
print(f"Ticker map loaded: {len(TICKER_MAP)} ISINs")

## Step 1 — Parse PDF and seed lot ledger

In [3]:
_tx_df, hld_df = parse_db_pdf(PDF_PATH)
lots_df = initialize_lots_from_holdings(hld_df)

print(f"Holdings parsed : {len(hld_df)} positions")
print(f"Lots seeded     : {len(lots_df)} lots")
hld_df[["isin", "wkn", "asset_name", "quantity", "cost_basis_eur", "market_value"]]

ImportError: pdfplumber is required for PDF parsing: pip install pdfplumber  (run inside Docker, not the host Python)

## Step 2 — Fetch live prices

In [ ]:
price_provider = YahooPriceProvider(
    isin_to_ticker=TICKER_MAP,
    fx_provider=ECBProvider(),
)

isins = lots_df["isin"].tolist()
PRICES_EUR = fetch_current_prices(isins, price_provider, REPORTING_DATE)

print(f"Prices fetched: {len(PRICES_EUR)}/{len(isins)} ISINs")
price_series = pd.Series(PRICES_EUR, name="price_eur").sort_index()
price_series

## Step 3 — Portfolio snapshot

Market value, unrealised gain, and gain % for every position at today's prices.

In [ ]:
snapshot_rows = []
for _, lot in lots_df.iterrows():
    isin   = lot["isin"]
    shares = float(lot["remaining_shares"])
    basis  = float(lot["lot_price_eur"])
    price  = PRICES_EUR.get(isin)
    if price is None:
        continue
    mv          = shares * price
    cost        = shares * basis
    unr_gain    = mv - cost
    unr_gain_pct = (price - basis) / basis if basis > 0 else 0.0
    name = hld_df.loc[hld_df["isin"] == isin, "asset_name"].iloc[0]
    snapshot_rows.append({
        "isin": isin,
        "name": name[:28],
        "ticker": TICKER_MAP.get(isin, ""),
        "shares": shares,
        "basis_eur": basis,
        "price_eur": price,
        "market_value_eur": mv,
        "unrealised_gain_eur": unr_gain,
        "unrealised_gain_pct": unr_gain_pct,
    })

snap_df = pd.DataFrame(snapshot_rows).sort_values("market_value_eur", ascending=False)

total_mv   = snap_df["market_value_eur"].sum()
total_gain = snap_df["unrealised_gain_eur"].sum()
print(f"Total market value  : EUR {total_mv:>12,.0f}")
print(f"Total unrealised gain: EUR {total_gain:>11,.0f}  ({total_gain/total_mv*100:.1f}% of MV)")
print()

snap_df.style \
    .format({
        "shares": "{:,.1f}",
        "basis_eur": "€{:,.2f}",
        "price_eur": "€{:,.2f}",
        "market_value_eur": "€{:,.0f}",
        "unrealised_gain_eur": "€{:,.0f}",
        "unrealised_gain_pct": "{:+.1%}",
    }) \
    .background_gradient(subset=["unrealised_gain_pct"], cmap="RdYlGn", vmin=-0.5, vmax=2.0)

## Step 4 — Build shared bear scenarios

One set of bear cases is reused across all positions (the shape of each scenario
is fixed; the absolute prices differ per position).

In [ ]:
bear_cases = build_bear_recovery_cases(
    start=BEAR_START,
    end=BEAR_END,
    step=BEAR_STEP,
    recovery_multiplier=BEAR_RECOVERY_MULT,
    min_recovery_return=BEAR_MIN_RECOVERY,
    max_recovery_return=BEAR_MAX_RECOVERY,
    base_recovery_probability=BEAR_BASE_PROB,
    min_recovery_probability=BEAR_MIN_PROB,
)
bear_case_order = bear_cases["case"].tolist()
print(f"{len(bear_cases)} bear scenarios: {bear_cases['case'].iloc[0]} → {bear_cases['case'].iloc[-1]}")
bear_cases

## Step 5 — Run stop-loss simulation for every position

In [ ]:
results = {}   # isin → {comparison_df, stop_benchmark, bear_recovery_df, baseline, meta}

for _, lot in lots_df.iterrows():
    isin   = lot["isin"]
    shares = float(lot["remaining_shares"])
    basis  = float(lot["lot_price_eur"])
    price  = PRICES_EUR.get(isin)
    if price is None:
        continue

    baseline       = sell_today_baseline(shares, price, basis, CAPITAL_GAINS_TAX_RATE)
    stop_bench     = build_stop_benchmark(STOP_LOSS_DROPS, shares, price, basis, CAPITAL_GAINS_TAX_RATE)
    bear_rec_df    = build_bear_recovery_table(bear_cases, shares, price, basis, CAPITAL_GAINS_TAX_RATE)
    comparison_df  = compare_stop_reentry_vs_hold(
        stop_bench, bear_rec_df, shares, basis, CAPITAL_GAINS_TAX_RATE,
        reentry_slippage_from_bear_low=REENTRY_SLIPPAGE,
        transaction_cost_rate=TRANSACTION_COST_RATE,
        allow_fractional_reentry_shares=ALLOW_FRACTIONAL_SHARES,
    )

    results[isin] = {
        "comparison": comparison_df,
        "stop_benchmark": stop_bench,
        "bear_recovery": bear_rec_df,
        "baseline": baseline,
        "shares": shares,
        "basis": basis,
        "price": price,
        "name": hld_df.loc[hld_df["isin"] == isin, "asset_name"].iloc[0],
        "ticker": TICKER_MAP.get(isin, isin),
    }

print(f"Simulated {len(results)} positions")

## Step 6 — Portfolio ranking

For each position: the **best possible** advantage from any stop × scenario combination,
and the **10%-stop / -30%-bear** point estimate as a consistent cross-position comparison.

**Reading the table:** `adv_10pct_stop_30pct_bear` is the extra EUR you'd have
after selling at a 10% stop, re-entering at the -30% bear low + 5% slippage,
and selling again at the assumed recovery price — compared to just holding through.

In [ ]:
ranking_rows = []

REFERENCE_STOP = 0.10
REFERENCE_BEAR = "Bear -30%"

for isin, r in results.items():
    cmp  = r["comparison"]
    trig = cmp[cmp["stop_triggers"]]

    if trig.empty:
        best_adv = None
        best_stop = None
        best_bear = None
    else:
        best_idx = trig["stop_reentry_advantage_vs_hold_after_recovery"].idxmax()
        best_row = trig.loc[best_idx]
        best_adv  = best_row["stop_reentry_advantage_vs_hold_after_recovery"]
        best_stop = best_row["stop_loss_drop"]
        best_bear = best_row["bear_case"]

    # Point estimate: 10% stop × -30% bear
    ref_slice = cmp[
        (cmp["stop_loss_drop"].round(4) == round(REFERENCE_STOP, 4)) &
        (cmp["bear_case"] == REFERENCE_BEAR)
    ]
    ref_adv = ref_slice["stop_reentry_advantage_vs_hold_after_recovery"].iloc[0] if not ref_slice.empty else None

    unr_pct = (r["price"] - r["basis"]) / r["basis"] if r["basis"] > 0 else 0.0
    ranking_rows.append({
        "ticker": r["ticker"],
        "isin": isin,
        "shares": r["shares"],
        "basis_eur": r["basis"],
        "price_eur": r["price"],
        "market_value_eur": r["shares"] * r["price"],
        "unrealised_gain_pct": unr_pct,
        "adv_best_eur": best_adv,
        "best_stop": best_stop,
        "best_bear": best_bear,
        "adv_10pct_stop_30pct_bear": ref_adv,
    })

rank_df = pd.DataFrame(ranking_rows).sort_values("adv_10pct_stop_30pct_bear", ascending=False)

rank_df.style \
    .format({
        "shares":                    "{:,.1f}",
        "basis_eur":                 "€{:,.2f}",
        "price_eur":                 "€{:,.2f}",
        "market_value_eur":          "€{:,.0f}",
        "unrealised_gain_pct":       "{:+.1%}",
        "adv_best_eur":              "€{:+,.0f}",
        "best_stop":                 "{:.0%}",
        "adv_10pct_stop_30pct_bear": "€{:+,.0f}",
    }, na_rep="—") \
    .background_gradient(subset=["adv_10pct_stop_30pct_bear"], cmap="RdYlGn")

## Step 7 — Per-position heatmaps

For each position: the stop+re-entry advantage vs. hold heatmap (rows = stop level,
columns = bear scenario).  Green = stop beats hold; red = hold beats stop.

Positions are shown in descending market-value order.

In [ ]:
def _advantage_heatmap(ax, comparison_df, bear_case_order, title, subtitle):
    """Draw one stop-reentry advantage heatmap on *ax*."""
    hm = comparison_df.pivot_table(
        index="stop_loss_drop",
        columns="bear_case",
        values="stop_reentry_advantage_vs_hold_after_recovery_pct",
        aggfunc="first",
    ).reindex(columns=bear_case_order)

    max_abs = max(np.nanmax(np.abs(hm.values)), 0.01)
    im = ax.imshow(hm.values, cmap="RdYlGn", vmin=-max_abs, vmax=max_abs, aspect="auto")

    # Annotate cells
    for i in range(hm.shape[0]):
        for j in range(hm.shape[1]):
            val = hm.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:+.0%}", ha="center", va="center", fontsize=6.5,
                        color="black" if abs(val) < 0.5 * max_abs else "white")

    ax.set_xticks(range(len(hm.columns)))
    ax.set_xticklabels(hm.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(hm.index)))
    ax.set_yticklabels([f"{v:.0%}" for v in hm.index], fontsize=8)
    ax.set_xlabel("Bear scenario", fontsize=9)
    ax.set_ylabel("Stop trigger", fontsize=9)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.text(0.5, 1.02, subtitle, transform=ax.transAxes, ha="center", fontsize=8, style="italic")
    return im


# Sort positions by market value descending
ordered_isins = (
    pd.Series({isin: r["shares"] * r["price"] for isin, r in results.items()})
    .sort_values(ascending=False)
    .index.tolist()
)

n = len(ordered_isins)
ncols = 2
nrows = (n + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 5.5 * nrows))
axes_flat = axes.flatten() if n > 1 else [axes]

for idx, isin in enumerate(ordered_isins):
    r   = results[isin]
    ax  = axes_flat[idx]
    unr = (r["price"] - r["basis"]) / r["basis"]
    title    = f"{r['ticker']}  ({isin})"
    subtitle = (
        f"{r['shares']:,.0f} shares | basis €{r['basis']:,.2f} | "
        f"price €{r['price']:,.2f} | gain {unr:+.1%} | "
        f"MV €{r['shares']*r['price']:,.0f}"
    )
    _advantage_heatmap(ax, r["comparison"], bear_case_order, title, subtitle)

# Hide any unused axes
for idx in range(len(ordered_isins), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle(
    "Stop+Re-entry Advantage vs Hold After Recovery — Real Portfolio\n"
    f"Tax rate {CAPITAL_GAINS_TAX_RATE:.3%} | Slippage {REENTRY_SLIPPAGE:.0%} | "
    f"Txn cost {TRANSACTION_COST_RATE:.1%}",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()

## Step 8 — Best stop per bear case (portfolio view)

For each position and each bear scenario: what is the best-available stop level
and the resulting advantage?  Answers: *"if the market falls 20%, which position
should I have had a stop on, and where?"*

In [ ]:
best_rows = []
for isin, r in results.items():
    cmp = r["comparison"]
    for bear_case in bear_case_order:
        slice_df = cmp[(cmp["bear_case"] == bear_case) & cmp["stop_triggers"]]
        if slice_df.empty:
            continue
        best = slice_df.loc[
            slice_df["stop_reentry_advantage_vs_hold_after_recovery"].idxmax()
        ]
        best_rows.append({
            "ticker":    r["ticker"],
            "isin":      isin,
            "bear_case": bear_case,
            "best_stop": best["stop_loss_drop"],
            "advantage_eur": best["stop_reentry_advantage_vs_hold_after_recovery"],
            "advantage_pct": best["stop_reentry_advantage_vs_hold_after_recovery_pct"],
        })

best_df = pd.DataFrame(best_rows)

# Pivot: rows = ticker, columns = bear case
pivot_adv = best_df.pivot_table(
    index="ticker", columns="bear_case",
    values="advantage_eur", aggfunc="first"
).reindex(columns=bear_case_order)

# Sort by the -30% bear scenario
SORT_COL = "Bear -30%"
if SORT_COL in pivot_adv.columns:
    pivot_adv = pivot_adv.sort_values(SORT_COL, ascending=False)

pivot_adv.style \
    .format("€{:+,.0f}", na_rep="—") \
    .background_gradient(cmap="RdYlGn", axis=None)

## Step 9 — Optimal stop level by position (heatmap)

Rows = positions (sorted by MV), columns = bear scenarios.
Cell value = the **best stop trigger** (% drop) for that position × scenario combination.
An empty cell means no stop would trigger in that scenario.

In [ ]:
pivot_stop = best_df.pivot_table(
    index="ticker", columns="bear_case",
    values="best_stop", aggfunc="first"
).reindex(columns=bear_case_order)

# Keep same row order as pivot_adv
pivot_stop = pivot_stop.reindex(index=pivot_adv.index)

fig, ax = plt.subplots(figsize=(14, 0.55 * len(pivot_stop) + 2))
vals = pivot_stop.values.astype(float)  # NaN for missing
im = ax.imshow(vals, cmap="RdYlGn_r", vmin=0.05, vmax=0.30, aspect="auto")

for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        v = vals[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=8)

ax.set_xticks(range(len(pivot_stop.columns)))
ax.set_xticklabels(pivot_stop.columns, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(pivot_stop.index)))
ax.set_yticklabels(pivot_stop.index, fontsize=9)
ax.set_title("Optimal Stop Trigger by Position × Bear Scenario", fontsize=11, fontweight="bold")
ax.set_xlabel("Bear scenario")
ax.set_ylabel("Position")
cbar = fig.colorbar(im, ax=ax, ticks=[0.05, 0.10, 0.15, 0.20, 0.25, 0.30])
cbar.ax.set_yticklabels(["5%", "10%", "15%", "20%", "25%", "30%"])
plt.tight_layout()
plt.show()